In [ ]:
import torch
import torch.nn as nn

import kagglehub

from torchvision.models import resnet18, resnet34, resnet50, resnet101, resnet152
from torchvision.models import ResNet18_Weights, ResNet34_Weights, ResNet50_Weights, ResNet101_Weights, ResNet152_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score

from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF


import pandas as pd
import numpy as np
from PIL import Image

import os
from pathlib import Path

import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
PLATE_MAX_LENGTH = 7 # AB12 CDE or any diplomatic plate
CHARACTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
NUM_CLASSES = len(CHARACTERS) + 1  # +1 for "No Character"

# Dataset

TODO:
- [X] Auto-download using Kaggle
- [ ] Use real world images rather than synthetic images

In [ ]:
# Download latest version
dataset_path = kagglehub.dataset_download("saadbinmunir/uk-licence-plate-synthetic-images")

In [ ]:
DATASET_FOLDER = Path(dataset_path) / "UKLicencePlateDataset"

TARGET_FOLDERS = ["whiteplate_normal", "yellowplate_normal"]

TARGET_FOLDERS = [DATASET_FOLDER / folder for folder in TARGET_FOLDERS]
FILE_TYPES = ["*.png", "*.jpg", "*.jpeg"]

# Get a list of the image files in the target folders
def get_image_files(target_folders):
    image_files = []
    for folder in target_folders:
        if not folder.exists():
            raise FileNotFoundError(f"Folder {folder} does not exist.")
        
        for file_type in FILE_TYPES:
            image_files.extend(list(folder.glob(file_type)))
        
    return image_files

sample_paths = get_image_files(TARGET_FOLDERS)

# Shuffle the sample paths with a seed for reproducibility
np.random.seed(42)
sample_paths = np.random.permutation(sample_paths)

# Print the number of images found
print(len(sample_paths), "images found in the dataset.")

# Create a DataFrame with the sample paths and the corresponding plates
plate_paths = {
    "path": sample_paths,
    "plate": [path.stem for path in sample_paths]  # Use the filename without extension as the plate
}

df = pd.DataFrame(plate_paths)

# Split the DataFrame into training, validation, and test sets
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.3, random_state=42)

print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

In [ ]:
def letter_to_index(letter: str):
    """
    Convert a letter to an index.
    The 0th index is reserved for "No Character".
    """
    if letter is None:
        return 0
    if letter not in CHARACTERS:
        raise ValueError(f"Invalid character: {letter}")
    return CHARACTERS.index(letter) + 1  # +1 because 0 is reserved

def plate_to_indices(plate: str):
    plate = plate.replace(" ", "").replace("-", "").upper()
    
    if len(plate) > PLATE_MAX_LENGTH:
        raise ValueError(f"Plate {plate} is too long.")
    
    indices = [letter_to_index(letter) for letter in plate]
    
    while len(indices) < PLATE_MAX_LENGTH:
        indices.append(0)  # Padding with "No Character"
    
    return torch.tensor(indices, dtype=torch.long)  # Shape: [PLATE_MAX_LENGTH]

def indices_to_plate(indices):
    chars = []
    for idx in indices:
        if idx == 0:
            chars.append('-')  # padding shown as dash
        else:
            chars.append(CHARACTERS[idx - 1])
    return "".join(chars)

print(plate_to_indices("AB12 CDE"))  # Example usage
print(plate_to_indices("1 ABC z  s"))   # Example usage

print(indices_to_plate(plate_to_indices("AB12 CDE")))  # Example usage


In [ ]:
class LicencePlateDataset(Dataset):
    def __init__(self, df, plate_to_indices_fn, transform=None):
        """
        df: pandas DataFrame with 'path' and 'plate' columns
        plate_to_indices_fn: function to convert plate string to indices tensor
        transform: image transformations to apply
        """
        self.df = df
        self.plate_to_indices = plate_to_indices_fn
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['path']
        plate = row['plate']
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        
        # Apply transforms if any
        if self.transform:
            image = self.transform(image)
        
        # Convert plate to indices tensor
        plate_indices = self.plate_to_indices(plate)
        
        return image, plate_indices

In [ ]:
# Boilerplate for pretrained ResNet models
boilerplate = [
    transforms.Resize((224, 224)),   # Resize images to 224x224 (ResNet input size)
    transforms.ToTensor(),           # Convert PIL image to tensor (C,H,W) in [0,1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225]     # ImageNet std
    )
]

# Define the base transform for the dataset
transform = transforms.Compose(boilerplate)

# Define the augmented transform for the dataset
transform_augmentated = transforms.Compose([
    # Rotate the image by a random angle between -10 and 10 degrees
    transforms.RandomRotation(degrees=60, expand=True),
    
    # Blur the image with a random radius between 0 and 50
    transforms.GaussianBlur(kernel_size=31, sigma=(0.1, 50)),
    
    # Noise the image with a random amount of noise
    transforms.Lambda(lambda x: TF.adjust_brightness(x, brightness_factor=np.random.uniform(0.1, 1))),
    
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2), shear=10),

    transforms.RandomPerspective(distortion_scale=0.1, p=0.5, fill=0),

    transforms.ColorJitter(
        brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1
    ),

    transforms.RandomAdjustSharpness(
        sharpness_factor=np.random.uniform(0.5, 2.0), p=0.5
    ),

    # Zoom the image by a random factor between 0.8 and 1.2
    # transforms.RandomResizedCrop(size=(224, 224), scale=(0.2, 1), ratio=(0.75, 1.3333)),
] + boilerplate)

train_dataset = LicencePlateDataset(train_df, plate_to_indices, transform=transform_augmentated)
val_dataset = LicencePlateDataset(val_df, plate_to_indices, transform=transform)
test_dataset = LicencePlateDataset(test_df, plate_to_indices, transform=transform)

LOADER_KARGS = {
    'batch_size': 32,
    'shuffle': True,
    'num_workers': 4
}

train_loader = DataLoader(train_dataset, **LOADER_KARGS)
val_loader = DataLoader(val_dataset, **LOADER_KARGS)
test_loader = DataLoader(test_dataset, **LOADER_KARGS)

In [ ]:
def show_sample_images(dataloader, num_samples=5, title="Sample Images"):
    """
    Show a few sample images from the dataloader.
    """
    # Sample one batch from dataloader
    images, plates = next(iter(dataloader))

    plt.figure(figsize=(15, 5))

    for i in range(num_samples):
        img = images[i]
        # Undo normalization for display: (img * std) + mean
        img = TF.to_pil_image(
            img * torch.tensor([0.229, 0.224, 0.225]).view(3,1,1) + 
            torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        )
        
        plate_str = indices_to_plate(plates[i].tolist())
        
        plt.subplot(1, num_samples, i+1)
        plt.imshow(img)
        plt.title(plate_str)
        plt.axis('off')

    plt.suptitle(title)
    plt.show()

show_sample_images(train_loader, num_samples=5, title="Train Sample Images")
show_sample_images(val_loader, num_samples=5, title="Validation Sample Images")
show_sample_images(test_loader, num_samples=5, title="Test Sample Images")

# Model

In [ ]:
class MultiLabelResNet(nn.Module):
    def __init__(self, resnet_base, num_labels : int, classes_per_label : int):
        super().__init__()

        # Get the base ResNet model
        self.resnet = resnet_base

        # Get the number of features from the last fully connected layer
        num_features = self.resnet.fc.in_features

        self.resnet.fc = torch.nn.Identity()  # Remove the original fully connected layer

        # Create the N vectors per label
        self.fcs = []
        for i in range(num_labels):
            fc = torch.nn.Linear(num_features, classes_per_label)
            self.fcs.append(fc)
    
        self.fcs = torch.nn.ModuleList(self.fcs)  # Convert to ModuleList for proper handling by PyTorch
    
    def forward(self, x):
        # Pass the input through the ResNet base model
        features = self.resnet(x)

        # Collect outputs from each fully connected layer
        outputs = []
        for fc in self.fcs:
            outputs.append(fc(features))
        
        return outputs

In [ ]:
base_model = resnet18(weights=ResNet18_Weights.DEFAULT) # TODO try other models
plate_model = MultiLabelResNet(base_model, PLATE_MAX_LENGTH, NUM_CLASSES)

## Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plate_model.to(device)

print(device, "device is available.")

In [ ]:
def evaluate_model(y_true, y_pred, num_positions=7):
    """
    Evaluate a multi-output, multi-class classifier (e.g. license plate characters).

    Parameters:
        y_true (array): Ground truth labels, shape (n_samples, num_positions)
        y_pred (array): Predicted labels, shape (n_samples, num_positions)
        num_positions (int): Number of character positions (default 7)

    Returns:
        dict with per-position and overall metrics
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    metrics = {
        'accuracy': [],
        'balanced_accuracy': [],
        'f1_macro': [],
        'cohen_kappa': []
    }

    for i in range(num_positions):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        metrics['accuracy'].append(accuracy_score(yt, yp))
        metrics['balanced_accuracy'].append(balanced_accuracy_score(yt, yp))
        metrics['f1_macro'].append(f1_score(yt, yp, average='macro'))
        metrics['cohen_kappa'].append(cohen_kappa_score(yt, yp))

    mean_metrics = {
        k + '_mean': np.mean(v) for k, v in metrics.items()
    }

    return {
        'per_position': metrics,
        'mean': mean_metrics
    }

def train(model, train_loader, val_loader, optimizer, device, num_epochs=10, criterion=nn.CrossEntropyLoss()):
    for epoch in range(num_epochs):
        model.train()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch")
        train_loss = 0.0

        # ===== Train =====
        for inputs, targets in pbar:
            inputs = inputs.to(device)
            targets = targets.to(device)  # shape: [batch_size, 7]

            optimizer.zero_grad()  # Reset gradients
            outputs = model(inputs)  # list of 7 outputs, each [B, num_classes]

            # Calculate loss for each character position
            loss = sum(criterion(output, targets[:, i]) for i, output in enumerate(outputs)) # Calculate total loss across all character positions
            loss.backward()  # Backpropagate the total loss

            optimizer.step()  # Update model parameters
            
            # Logging
            train_loss += loss.item()
            pbar.set_postfix(loss=loss.item())

        print(f"Epoch {epoch + 1}: Avg Training loss = {train_loss / len(train_loader):.4f}")

        # ===== Evaluate =====
        model.eval()
        y_true_all, y_pred_all = [], []

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                targets = targets.to(device)
                outputs = model(inputs)  # list of 7 outputs

                preds = [torch.argmax(output, dim=1).cpu().numpy() for output in outputs]  # list of 7 [B]
                y_pred_batch = np.stack(preds, axis=1)  # [B, 7]
                y_true_batch = targets.cpu().numpy()    # [B, 7]

                y_pred_all.append(y_pred_batch)
                y_true_all.append(y_true_batch)

        # Stack all batches
        y_true_all = np.vstack(y_true_all)
        y_pred_all = np.vstack(y_pred_all)

        metrics = evaluate_model(y_true_all, y_pred_all)

        print(f"Validation F1 Macro (mean): {metrics['mean']['f1_macro_mean']:.3f}")
        print(f"Validation Accuracy (mean): {metrics['mean']['accuracy_mean']:.3f}")
        print(f"Validation Balanced Accuracy (mean): {metrics['mean']['balanced_accuracy_mean']:.3f}")
        print(f"Validation Cohen Kappa (mean): {metrics['mean']['cohen_kappa_mean']:.3f}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(plate_model.parameters(), lr=1e-4)

train(model=plate_model,
      train_loader=train_loader,
      val_loader=val_loader,
      optimizer=optimizer,
      device=device,
      num_epochs=10,
      criterion=criterion
)

In [ ]:
# Export plate_model to a .pt
torch.save(plate_model.state_dict(), "plate_model.pt")